# State Space Models — from Vanilla SSM to Mamba

This notebook is a self-contained walkthrough of the `ssm-demo` project.  
Every implementation is defined inline — no local package install required.

**Contents**
1. [Setup & imports](#1-setup--imports)
2. [Vanilla SSM](#2-vanilla-ssm)
3. [Selective SSM (S6)](#3-selective-ssm-s6)
4. [Mamba Block](#4-mamba-block)
5. [Full Mamba Language Model](#5-full-mamba-language-model)
6. [Demos](#6-demos)
7. [Mini Training Loop](#7-mini-training-loop)

---
## 1. Setup & imports

Install dependencies if they are not already present, then import everything needed.

In [ ]:
%pip install --quiet torch numpy einops

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"PyTorch {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

---
## 2. Vanilla SSM

A linear State Space Model (SSM) maps an input sequence $u(t)$ to an output sequence $y(t)$
through a hidden state $h(t)$:

$$h'(t) = A\,h(t) + B\,u(t) \qquad y(t) = C\,h(t) + D\,u(t)$$

In the **discrete-time** (recurrent) form using the Zero-Order-Hold (ZOH) rule with step size $\Delta$:

$$h_t = \bar{A}\,h_{t-1} + \bar{B}\,u_t \qquad y_t = C\,h_t + D\,u_t$$

where $\bar{A} = \exp(\Delta A)$ and $\bar{B} = (\bar{A} - I)\,A^{-1}\,B$.

$A$ and $B$ are kept **diagonal** (S4D parameterisation) for efficiency.

In [ ]:
class SSM(nn.Module):
    """Learnable discrete-time diagonal SSM.

    Parameters
    ----------
    d_model : int
        Input/output feature dimension.
    d_state : int
        Dimension of the hidden state (N in the S4 paper).
    dt_min, dt_max : float
        Range for the learnable log time-step Δ (log-uniform init).
    """

    def __init__(
        self,
        d_model: int,
        d_state: int = 16,
        dt_min: float = 1e-3,
        dt_max: float = 0.1,
    ) -> None:
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state

        # A: diagonal state matrix, parameterised as log(|A|) so A stays
        # real-negative and the system is stable.
        # Shape: (d_model, d_state)
        A = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0)
        A = A.expand(d_model, -1)           # (d_model, d_state)
        self.A_log = nn.Parameter(torch.log(A))

        # B and C: input / output projection onto the state space
        # Shape: (d_model, d_state)
        self.B = nn.Parameter(torch.randn(d_model, d_state) * 0.01)
        self.C = nn.Parameter(torch.randn(d_model, d_state) * 0.01)

        # D: skip connection (direct term)
        self.D = nn.Parameter(torch.ones(d_model))

        # Δ (delta): per-feature log time-step, log-uniform in [dt_min, dt_max]
        log_dt = (
            torch.rand(d_model) * (math.log(dt_max) - math.log(dt_min))
            + math.log(dt_min)
        )
        self.log_dt = nn.Parameter(log_dt)

    def _get_AB_bar(self) -> tuple[torch.Tensor, torch.Tensor]:
        """Return discretised (Ā, B̄) using the ZOH rule for diagonal A."""
        dt = torch.exp(self.log_dt)                     # (d_model,)
        A = -torch.exp(self.A_log)                      # (d_model, d_state) negative

        # Ā = exp(Δ A)
        A_bar = torch.exp(dt.unsqueeze(-1) * A)         # (d_model, d_state)

        # B̄ = (Ā − I) / A * B  (element-wise for diagonal A)
        B_bar = (A_bar - 1.0) / A * self.B              # (d_model, d_state)

        return A_bar, B_bar

    def forward(self, u: torch.Tensor) -> torch.Tensor:
        """Run the SSM over a sequence.

        Parameters
        ----------
        u : torch.Tensor, shape (batch, seq_len, d_model)

        Returns
        -------
        y : torch.Tensor, shape (batch, seq_len, d_model)
        """
        B_batch, L, d = u.shape
        assert d == self.d_model, f"Expected d_model={self.d_model}, got {d}"

        A_bar, B_bar = self._get_AB_bar()   # (d_model, d_state) each
        C = self.C                           # (d_model, d_state)
        D = self.D                           # (d_model,)

        # Initialise hidden state
        h = torch.zeros(B_batch, d, self.d_state, device=u.device, dtype=u.dtype)

        ys = []
        for t in range(L):
            u_t = u[:, t, :]                             # (batch, d_model)
            h = A_bar.unsqueeze(0) * h + B_bar.unsqueeze(0) * u_t.unsqueeze(-1)
            y_t = (h * C.unsqueeze(0)).sum(-1)           # (batch, d_model)
            y_t = y_t + D * u_t                          # skip connection
            ys.append(y_t)

        return torch.stack(ys, dim=1)                    # (batch, L, d_model)

---
## 3. Selective SSM (S6)

The key innovation in **Mamba** is making $B$, $C$, and $\Delta$ **input-dependent** (selective), allowing the model to selectively propagate or filter information at each time step.

Selective scan:
$$\bar{A}_t = \exp(\Delta_t \cdot A), \quad \bar{B}_t = \Delta_t \cdot B_t$$
$$h_t = \bar{A}_t \cdot h_{t-1} + \bar{B}_t \cdot x_t, \quad y_t = C_t \cdot h_t + D \cdot x_t$$

> Gu & Dao, *"Mamba: Linear-Time Sequence Modeling with Selective State Spaces"*, 2023 — <https://arxiv.org/abs/2312.00752>

In [ ]:
class SelectiveSSM(nn.Module):
    """The S6 core of Mamba: a selective state-space scan.

    Parameters
    ----------
    d_inner : int
        Inner channel dimension (= expand * d_model).
    d_state : int
        SSM state dimension N.
    dt_rank : int
        Rank of the Δ projection.  Mamba uses ceil(d_model / 16).
    dt_min, dt_max : float
        Log-uniform range for Δ initialisation.
    """

    def __init__(
        self,
        d_inner: int,
        d_state: int = 16,
        dt_rank: int | None = None,
        dt_min: float = 1e-3,
        dt_max: float = 0.1,
    ) -> None:
        super().__init__()
        self.d_inner = d_inner
        self.d_state = d_state
        self.dt_rank = dt_rank if dt_rank is not None else math.ceil(d_inner / 16)

        # A: fixed structure, learned as log|A| (kept negative / stable)
        A = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0)
        A = A.expand(d_inner, -1)                       # (d_inner, d_state)
        self.A_log = nn.Parameter(torch.log(A))

        # D: skip-connection weight
        self.D = nn.Parameter(torch.ones(d_inner))

        # Projections that produce (Δ, B, C) from the input
        # x_proj maps d_inner → dt_rank + 2*d_state
        self.x_proj = nn.Linear(d_inner, self.dt_rank + 2 * d_state, bias=False)

        # dt_proj maps dt_rank → d_inner (with bias for Δ init)
        self.dt_proj = nn.Linear(self.dt_rank, d_inner, bias=True)

        # Initialise dt_proj so Δ ≈ softplus⁻¹(U[dt_min, dt_max])
        dt_init = (
            torch.rand(d_inner) * (math.log(dt_max) - math.log(dt_min))
            + math.log(dt_min)
        )
        dt_init = torch.exp(dt_init)                    # (d_inner,)
        inv_softplus = torch.log(torch.expm1(dt_init))  # inverse softplus
        with torch.no_grad():
            self.dt_proj.bias.copy_(inv_softplus)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Selective scan over the sequence.

        Parameters
        ----------
        x : torch.Tensor, shape (B, L, d_inner)

        Returns
        -------
        y : torch.Tensor, shape (B, L, d_inner)
        """
        B_batch, L, _ = x.shape

        A = -torch.exp(self.A_log.float())              # (d_inner, d_state) negative

        # Compute input-dependent Δ, B, C
        x_dbl = self.x_proj(x)                         # (B, L, dt_rank + 2*d_state)
        dt_raw, B_mat, C_mat = x_dbl.split(
            [self.dt_rank, self.d_state, self.d_state], dim=-1
        )
        dt = F.softplus(self.dt_proj(dt_raw))           # (B, L, d_inner)

        # Discretise: ZOH for A, Euler for B
        dA = torch.exp(dt.unsqueeze(-1) * A)            # (B, L, d_inner, d_state)
        dB = dt.unsqueeze(-1) * B_mat.unsqueeze(2)      # (B, L, d_inner, d_state)

        # Sequential selective scan
        h = x.new_zeros(B_batch, self.d_inner, self.d_state)
        ys = []
        for t in range(L):
            h = dA[:, t] * h + dB[:, t] * x[:, t].unsqueeze(-1)
            y_t = (h * C_mat[:, t].unsqueeze(1)).sum(-1)  # (B, d_inner)
            ys.append(y_t)

        y = torch.stack(ys, dim=1)                      # (B, L, d_inner)
        y = y + x * self.D                              # skip connection
        return y

---
## 4. Mamba Block

A single **Mamba residual block** wraps `SelectiveSSM` with a gated MLP structure:

```
Input  x : (B, L, d_model)
       │
       ├── Linear ── d_inner ── Conv1d ── SiLU ── SelectiveSSM ──┐
       │                                                          ×  ── Linear ── Output
       └── Linear ── d_inner ── SiLU  ──────────────────────────────┘
                                                                  + residual
```

The causal depthwise `Conv1d` ensures that position $t$ cannot see future positions.

In [ ]:
class MambaBlock(nn.Module):
    """A single Mamba residual block.

    Parameters
    ----------
    d_model : int
        Model (input/output) dimension.
    d_state : int
        SSM state dimension N.
    d_conv : int
        Kernel size of the depthwise causal convolution.
    expand : int
        Channel expansion factor inside the block (d_inner = expand * d_model).
    """

    def __init__(
        self,
        d_model: int,
        d_state: int = 16,
        d_conv: int = 4,
        expand: int = 2,
    ) -> None:
        super().__init__()
        self.d_model = d_model
        self.d_inner = expand * d_model

        # Input norm + projection to 2×d_inner (two branches)
        self.norm = nn.LayerNorm(d_model)
        self.in_proj = nn.Linear(d_model, 2 * self.d_inner, bias=False)

        # Causal depthwise conv on the SSM branch
        self.conv1d = nn.Conv1d(
            in_channels=self.d_inner,
            out_channels=self.d_inner,
            kernel_size=d_conv,
            groups=self.d_inner,
            padding=d_conv - 1,       # left-pad for causality
            bias=True,
        )

        # Selective SSM
        self.ssm = SelectiveSSM(d_inner=self.d_inner, d_state=d_state)

        # Output projection back to d_model
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply Mamba block with residual connection.

        Parameters
        ----------
        x : torch.Tensor, shape (B, L, d_model)

        Returns
        -------
        out : torch.Tensor, shape (B, L, d_model)
        """
        residual = x
        x = self.norm(x)

        # Split into SSM branch (x_ssm) and gate branch (z)
        xz = self.in_proj(x)                            # (B, L, 2*d_inner)
        x_ssm, z = xz.chunk(2, dim=-1)                 # each (B, L, d_inner)

        # SSM branch: causal conv → SiLU → SelectiveSSM
        x_ssm = x_ssm.transpose(1, 2)                  # (B, d_inner, L)
        x_ssm = self.conv1d(x_ssm)[:, :, : x.shape[1]] # trim padding → causal
        x_ssm = x_ssm.transpose(1, 2)                  # (B, L, d_inner)
        x_ssm = F.silu(x_ssm)
        x_ssm = self.ssm(x_ssm)                        # (B, L, d_inner)

        # Gate branch
        z = F.silu(z)                                   # (B, L, d_inner)

        # Combine and project
        out = self.out_proj(x_ssm * z)                  # (B, L, d_model)
        return out + residual                           # residual connection

---
## 5. Full Mamba Language Model

Stack $N$ `MambaBlock` layers with a token embedding and a tied LM head:

```
Token IDs → Embedding → [MambaBlock] × N → LayerNorm → LM Head → Logits
```

Tying the embedding and output weights is a standard trick that reduces parameters and improves generalisation.

In [ ]:
class MambaModel(nn.Module):
    """Causal sequence model composed of stacked Mamba blocks.

    Parameters
    ----------
    vocab_size : int
        Vocabulary size (number of token embeddings).
    d_model : int
        Model dimension.
    n_layers : int
        Number of Mamba blocks to stack.
    d_state : int
        SSM state dimension for each block.
    d_conv : int
        Depthwise conv kernel size for each block.
    expand : int
        Inner expansion factor for each block.
    """

    def __init__(
        self,
        vocab_size: int,
        d_model: int = 128,
        n_layers: int = 4,
        d_state: int = 16,
        d_conv: int = 4,
        expand: int = 2,
    ) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)

        self.layers = nn.ModuleList(
            [
                MambaBlock(
                    d_model=d_model,
                    d_state=d_state,
                    d_conv=d_conv,
                    expand=expand,
                )
                for _ in range(n_layers)
            ]
        )

        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        # Tie embedding and output weights (standard in language models)
        self.lm_head.weight = self.embedding.weight

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        """Forward pass.

        Parameters
        ----------
        input_ids : torch.Tensor, shape (batch, seq_len)

        Returns
        -------
        logits : torch.Tensor, shape (batch, seq_len, vocab_size)
        """
        x = self.embedding(input_ids)       # (B, L, d_model)

        for layer in self.layers:
            x = layer(x)

        x = self.norm(x)
        return self.lm_head(x)              # (B, L, vocab_size)

    def count_parameters(self) -> int:
        """Return the total number of trainable parameters."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

---
## 6. Demos

### 6a. Vanilla SSM

In [ ]:
torch.manual_seed(42)

ssm = SSM(d_model=32, d_state=16)
x = torch.randn(4, 64, 32)        # batch=4, seq_len=64, d_model=32
y = ssm(x)

print(f"Input  shape : {x.shape}")
print(f"Output shape : {y.shape}")
print(f"Parameters   : {sum(p.numel() for p in ssm.parameters()):,}")

### 6b. Single Mamba block

In [ ]:
torch.manual_seed(42)

block = MambaBlock(d_model=64, d_state=16, d_conv=4, expand=2)
x = torch.randn(2, 128, 64)        # batch=2, seq_len=128, d_model=64
y = block(x)

print(f"Input  shape : {x.shape}")
print(f"Output shape : {y.shape}")
print(f"Parameters   : {sum(p.numel() for p in block.parameters()):,}")

### 6c. Full Mamba language model

In [ ]:
torch.manual_seed(42)

model = MambaModel(
    vocab_size=256,
    d_model=128,
    n_layers=4,
    d_state=16,
    d_conv=4,
    expand=2,
)
token_ids = torch.randint(0, 256, (2, 64))   # batch=2, seq_len=64
logits = model(token_ids)

print(f"Token IDs shape : {token_ids.shape}")
print(f"Logits shape    : {logits.shape}")
print(f"Parameters      : {model.count_parameters():,}")

---
## 7. Mini Training Loop

Next-token prediction with cross-entropy loss — 5 optimisation steps.

In [ ]:
torch.manual_seed(0)

model = MambaModel(vocab_size=64, d_model=64, n_layers=2, d_state=8)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for step in range(5):
    tokens = torch.randint(0, 64, (4, 32))      # (batch, seq_len)
    inputs, targets = tokens[:, :-1], tokens[:, 1:]

    logits = model(inputs)                       # (4, 31, 64)
    loss = F.cross_entropy(
        logits.reshape(-1, 64),
        targets.reshape(-1),
    )
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"step {step + 1}: loss = {loss.item():.4f}")

print("\nDone ✓")